# autoreduce analysis

Visualize the reduction progress from `results.tsv`.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
# Load results
df = pd.read_csv("results.tsv", sep="\t")
df["attempt"] = range(len(df))
print(f"Total attempts: {len(df)}")
print(f"Kept: {(df.status == 'keep').sum()}")
print(f"Discarded: {(df.status == 'discard').sum()}")
print(f"Failed: {(df.status == 'fail').sum()}")
print(f"Best score: {df[df.status.isin(['keep', 'baseline'])].score.min():.2f}")
print(f"Baseline score: {df.iloc[0].score:.2f}")

In [ ]:
# Compute running best
kept = df[df.status.isin(["keep", "baseline"])].copy()
kept["running_best"] = kept.score.cummin()

# Plot
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), gridspec_kw={"height_ratios": [3, 1]})

# Top: complexity over attempts
colors = {
    "keep": "#2ecc71",
    "discard": "#e74c3c",
    "fail": "#95a5a6",
    "baseline": "#3498db",
}
for status, group in df.groupby("status"):
    c = colors.get(status, "#333")
    marker = "o" if status == "keep" else "x" if status == "discard" else "s"
    valid = group[group.score > 0]
    ax1.scatter(
        valid.attempt,
        valid.score,
        c=c,
        marker=marker,
        s=30,
        label=status,
        alpha=0.7,
        zorder=3,
    )

# Running best line
ax1.step(
    kept.attempt,
    kept.running_best,
    where="post",
    color="#2ecc71",
    linewidth=2,
    label="running best",
    zorder=2,
)

ax1.set_ylabel("Composite Score")
ax1.set_title("autoreduce progress")
ax1.legend(loc="upper right")
ax1.grid(True, alpha=0.3)

# Bottom: delta per kept change
kept_only = df[df.status == "keep"].copy()
if len(kept_only) > 1:
    ax2.bar(kept_only.attempt, kept_only.delta, color="#2ecc71", alpha=0.7)
    ax2.set_ylabel("Delta (negative = improvement)")
    ax2.axhline(y=0, color="black", linewidth=0.5)
    ax2.grid(True, alpha=0.3)

ax2.set_xlabel("Attempt #")

plt.tight_layout()
plt.savefig("../progress.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved to progress.png")

In [ ]:
# Show kept reductions
print("\nKept reductions:")
print("=" * 80)
for _, row in kept.iterrows():
    delta_str = f"{row.delta:+.2f}" if row.delta != 0 else "baseline"
    print(f"  [{row.commit}] score={row.score:.2f} ({delta_str}) -- {row.description}")

In [ ]:
# Summary statistics
baseline = df.iloc[0].score
current_best = kept.score.min()
reduction_pct = (1 - current_best / baseline) * 100 if baseline > 0 else 0

print("\nSummary:")
print(f"  Baseline score:    {baseline:.2f}")
print(f"  Current best:      {current_best:.2f}")
print(f"  Total reduction:   {reduction_pct:.1f}%")
print(f"  Attempts:          {len(df)}")
print(f"  Success rate:      {(df.status == 'keep').sum() / len(df) * 100:.1f}%")
print(f"  Avg delta (kept):  {kept_only.delta.mean():.2f}" if len(kept_only) > 1 else "")